In [34]:
%%capture
%pip install -e ..

In [35]:
import os
import re
import yaml
import random
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sklearn.metrics import cohen_kappa_score
from sycophancy.utils import lookup_answer
from sycophancy.verdict_analyser import ResultsAnalyser
from sycophancy.sycophancy_analyser import SycophancyAnalyser

In [36]:
load_dotenv()
MODELS = ["gpt-oss:120b", "qwen3:4b"]
RUNS = ["run_1", "run_2", "run_3"]
DATA_DIR = Path().cwd().parent / "data"
ANNOTATION_DIR = DATA_DIR / "annotation"

## LLM-as-Judge

### Make sample

In [5]:
random.seed(42)

all_outputs = []
for model in MODELS:
    for run in RUNS:
        with open(DATA_DIR / f"answers_{model}_full_{run}.yaml", "r") as file:
            outputs = yaml.safe_load(file)
        all_outputs.extend(outputs.values())

sample = random.sample(all_outputs, 103) # Will take three of those for few shot

final_sample = []
for element in sample:
    answers = [element["answer_positive"], element["answer_neutral"], element["answer_negative"]]
    final_sample.append((element["question"], random.choice(answers)))

double_sample = final_sample[:100] # Take only 100 for the final sample
few_shot_sample = final_sample[100:] # Take the last three for few shot

df = pd.DataFrame(double_sample, columns=["question", "advice"])
df["binary_validation_annotator_1"] = None
df["binary_validation_annotator_2"] = None
df["likert_validation_annotator_1"] = None
df["likert_validation_annotator_2"] = None
df["binary_framing_annotator_1"] = None
df["binary_framing_annotator_2"] = None
df["likert_framing_annotator_1"] = None
df["likert_framing_annotator_2"] = None
df.to_csv(ANNOTATION_DIR / "sample_100.csv", index=False)

df_few_shot = pd.DataFrame(few_shot_sample, columns=["question", "advice"])
df_few_shot["binary_validation"] = None
df_few_shot["likert_validation"] = None
df_few_shot["binary_framing"] = None
df_few_shot["likert_framing"] = None
df_few_shot.to_csv(ANNOTATION_DIR / "few_shot_3.csv", index=False)


### Agreement rates

In [37]:
def agreement_rate(column_1, column_2):
    pairwise_removal = column_1.notna() & column_2.notna()
    return (column_1[pairwise_removal] == column_2[pairwise_removal]).mean()

def cohens_kappa(column_1, column_2, annotation_type="binary"):
    pairwise_removal = column_1.notna() & column_2.notna()
    if annotation_type == "binary":
        return cohen_kappa_score(column_1[pairwise_removal], column_2[pairwise_removal])
    else:
        return cohen_kappa_score(column_1[pairwise_removal], column_2[pairwise_removal], weights="linear")


### Double annotation

In [ ]:
with open(ANNOTATION_DIR / "sample_100.csv", "r") as file:
    sample_100 = pd.read_csv(file)

with open(ANNOTATION_DIR / "sample_100_aurelie.tsv", "r") as file:
    sample_100_annotator_1 = pd.read_csv(file, sep="\t")

with open(ANNOTATION_DIR / "sample_100_sharleen.tsv", "r") as file:
    sample_100_annotator_2 = pd.read_csv(file, sep="\t")

sample_100 = sample_100[["question", "advice"]]
sample_100_annotator_1 = sample_100_annotator_1[["binary_validation_annotator_1", "binary_framing_annotator_1", "likert_validation_annotator_1", "likert_framing_annotator_1"]]
sample_100_annotator_2 = sample_100_annotator_2[["binary_validation_annotator_2", "binary_framing_annotator_2", "likert_validation_annotator_2", "likert_framing_annotator_2"]]
sample_100 = pd.concat([sample_100, sample_100_annotator_1], axis=1)
sample_100 = pd.concat([sample_100, sample_100_annotator_2], axis=1)
sample_100.to_csv(ANNOTATION_DIR / "sample_100_double_annotated.csv", index=False)

### Double agreement

In [43]:
with open(ANNOTATION_DIR / "sample_100_double_annotated.csv", "r") as file:
    double_sample_100 = pd.read_csv(file)

In [44]:
# Agreement
binary_validation_agreement = agreement_rate(
    double_sample_100["binary_validation_annotator_1"],
    double_sample_100["binary_validation_annotator_2"]
)
likert_validation_agreement = agreement_rate(
    double_sample_100["likert_validation_annotator_1"],
    double_sample_100["likert_validation_annotator_2"]
)
binary_framing_agreement = agreement_rate(
    double_sample_100["binary_framing_annotator_1"],
    double_sample_100["binary_framing_annotator_2"]
)
likert_framing_agreement = agreement_rate(
    double_sample_100["likert_framing_annotator_1"],
    double_sample_100["likert_framing_annotator_2"]
)

print(f"Agreement rate for binary framing: {binary_framing_agreement * 100:.2f}%")
print(f"Agreement rate for likert framing: {likert_framing_agreement * 100:.2f}%")
print(f"Agreement rate for binary validation: {binary_validation_agreement * 100:.2f}%")
print(f"Agreement rate for likert validation: {likert_validation_agreement * 100:.2f}%")

Agreement rate for binary framing: 75.00%
Agreement rate for likert framing: 29.00%
Agreement rate for binary validation: 82.00%
Agreement rate for likert validation: 40.00%


In [45]:
# Kappa
binary_validation_kappa = cohens_kappa(
    double_sample_100["binary_validation_annotator_1"],
    double_sample_100["binary_validation_annotator_2"],
    annotation_type="binary"
)
likert_validation_kappa = cohens_kappa(
    double_sample_100["likert_validation_annotator_1"],
    double_sample_100["likert_validation_annotator_2"],
    annotation_type="likert"
)
binary_framing_kappa = cohens_kappa(
    double_sample_100["binary_framing_annotator_1"],
    double_sample_100["binary_framing_annotator_2"],
    annotation_type="binary"
)
likert_framing_kappa = cohens_kappa(
    double_sample_100["likert_framing_annotator_1"],
    double_sample_100["likert_framing_annotator_2"],
    annotation_type="likert"
)

print(f"Cohen's kappa for binary framing: {binary_framing_kappa:.2f}")
print(f"Cohen's kappa for likert framing: {likert_framing_kappa:.2f}")
print(f"Cohen's kappa for binary validation: {binary_validation_kappa:.2f}")
print(f"Cohen's kappa for likert validation: {likert_validation_kappa:.2f}")

Cohen's kappa for binary framing: 0.39
Cohen's kappa for likert framing: 0.28
Cohen's kappa for binary validation: 0.57
Cohen's kappa for likert validation: 0.42


### LLM annotation

In [46]:
with open(ANNOTATION_DIR / "sample_100_double_annotated.csv", "r") as file:
    sample_100 = pd.read_csv(file)

In [ ]:
sycophancy_analyser = SycophancyAnalyser(api_key = os.getenv("OLLAMA_API_KEY"))
answers_framing = sycophancy_analyser.llm_judge_test_framing(sample_100)
answers_validation = sycophancy_analyser.llm_judge_test_validation(sample_100)

100%|██████████| 100/100 [09:20<00:00,  5.61s/it]


In [ ]:
answers_df = []
for answer_framing, answers_validation in zip(answers_framing, answers_validation):
    answers_df.append({
        "binary_validation_llm": lookup_answer(answers_validation, "Binary Validation Sycophancy"),
        "likert_validation_llm": lookup_answer(answers_validation, "Likert Validation Sycophancy"),
        "binary_framing_llm": lookup_answer(answer_framing, "Binary Framing Sycophancy"),
        "likert_framing_llm": lookup_answer(answer_framing, "Likert Framing Sycophancy")
    })

answers_df = pd.DataFrame(answers_df)
answers_df.head(5)


,binary_validation_llm,likert_validation_llm,binary_framing_llm,likert_framing_llm
0,1.0,5.0,1,5
1,1.0,5.0,1,5
2,1.0,4.0,1,5
3,1.0,4.0,1,5
4,1.0,4.0,1,5


In [ ]:
LLM_judge_df = pd.concat([sample_100, answers_df], axis=1)
LLM_judge_df.to_csv(ANNOTATION_DIR / "sample_100_with_llm_judge.csv", index=False)

# TEMPORARY

In [47]:
with open(ANNOTATION_DIR / "sample_100_with_llm_judge.csv", "r") as file:
    final_sample_df = pd.read_csv(file)

with open(ANNOTATION_DIR / "sample_100_double_annotated.csv", "r") as file:
    double_sample_df = pd.read_csv(file)

final_sample_df["binary_validation_annotator_1"] = double_sample_df["binary_validation_annotator_1"]
final_sample_df["likert_validation_annotator_1"] = double_sample_df["likert_validation_annotator_1"]
final_sample_df["binary_framing_annotator_1"] = double_sample_df["binary_framing_annotator_1"]
final_sample_df["likert_framing_annotator_1"] = double_sample_df["likert_framing_annotator_1"]

final_sample_df["binary_validation_annotator_2"] = double_sample_df["binary_validation_annotator_2"]
final_sample_df["likert_validation_annotator_2"] = double_sample_df["likert_validation_annotator_2"]
final_sample_df["binary_framing_annotator_2"] = double_sample_df["binary_framing_annotator_2"]
final_sample_df["likert_framing_annotator_2"] = double_sample_df["likert_framing_annotator_2"]
final_sample_df.to_csv(ANNOTATION_DIR / "sample_100_with_llm_judge.csv", index=False)


### LLM agreement

In [48]:
with open(ANNOTATION_DIR / "sample_100_with_llm_judge.csv", "r") as file:
    final_sample_df = pd.read_csv(file)

In [49]:
# Agreement
binary_validation_agreement = agreement_rate(
    final_sample_df["binary_validation_annotator_1"],
    final_sample_df["binary_validation_llm"]
)
likert_validation_agreement = agreement_rate(
    final_sample_df["likert_validation_annotator_1"],
    final_sample_df["likert_validation_llm"]
)
binary_framing_agreement = agreement_rate(
    final_sample_df["binary_framing_annotator_1"],
    final_sample_df["binary_framing_llm"]
)
likert_framing_agreement = agreement_rate(
    final_sample_df["likert_framing_annotator_1"],
    final_sample_df["likert_framing_llm"]
)

print(f"Agreement rate for binary framing: {binary_framing_agreement * 100:.2f}%")
print(f"Agreement rate for likert framing: {likert_framing_agreement * 100:.2f}%")
print(f"Agreement rate for binary validation: {binary_validation_agreement * 100:.2f}%")
print(f"Agreement rate for likert validation: {likert_validation_agreement * 100:.2f}%")

Agreement rate for binary framing: 81.00%
Agreement rate for likert framing: 54.00%
Agreement rate for binary validation: 82.83%
Agreement rate for likert validation: 44.44%


In [50]:
# Kappa
binary_validation_kappa = cohens_kappa(
    final_sample_df["binary_validation_annotator_1"],
    final_sample_df["binary_validation_llm"],
    annotation_type="binary"
)
likert_validation_kappa = cohens_kappa(
    final_sample_df["likert_validation_annotator_1"],
    final_sample_df["likert_validation_llm"],
    annotation_type="likert"
)
binary_framing_kappa = cohens_kappa(
    final_sample_df["binary_framing_annotator_1"],
    final_sample_df["binary_framing_llm"],
    annotation_type="binary"
)
likert_framing_kappa = cohens_kappa(
    final_sample_df["likert_framing_annotator_1"],
    final_sample_df["likert_framing_llm"],
    annotation_type="likert"
)

print(f"Cohen's kappa for binary framing: {binary_framing_kappa:.2f}")
print(f"Cohen's kappa for likert framing: {likert_framing_kappa:.2f}")
print(f"Cohen's kappa for binary validation: {binary_validation_kappa:.2f}")
print(f"Cohen's kappa for likert validation: {likert_validation_kappa:.2f}")

Cohen's kappa for binary framing: 0.47
Cohen's kappa for likert framing: 0.43
Cohen's kappa for binary validation: 0.57
Cohen's kappa for likert validation: 0.46


## Full run

In [ ]:
for model in MODELS:
    for run in RUNS:
        print(f"RESULTS FOR MODEL: {model}, RUN: {run}\n")
        with open(DATA_DIR / f"answers_{model}_full_{run}.yaml", "r") as file:
            outputs = yaml.safe_load(file)

        sycophancy_analyser = SycophancyAnalyser(api_key = os.getenv("OLLAMA_API_KEY"), outputs=outputs)
        all_results_df = sycophancy_analyser.run_judge()
        all_results_df.to_csv(ANNOTATION_DIR / f"results_{model}_full_{run}.csv", index=False)

RESULTS FOR MODEL: gpt-oss:120b, RUN: run_1



100%|██████████| 150/150 [1:24:03<00:00, 33.62s/it]


RESULTS FOR MODEL: gpt-oss:120b, RUN: run_2



100%|██████████| 150/150 [1:23:13<00:00, 33.29s/it]


RESULTS FOR MODEL: gpt-oss:120b, RUN: run_3



100%|██████████| 150/150 [1:22:43<00:00, 33.09s/it]


... (2 outputs left out)

RESULTS FOR MODEL: qwen3:4b, RUN: run_2



100%|██████████| 150/150 [1:22:47<00:00, 33.12s/it]


RESULTS FOR MODEL: qwen3:4b, RUN: run_3



100%|██████████| 150/150 [1:23:27<00:00, 33.39s/it]
